# Analisi polarimetrica — tesi triennale

Notebook ordinato che orchestra la pipeline 2D (Stokes + retardance + UMAP) usando il package `polarimetro/`.

**Ordine pipeline obbligato** (vedi `CLAUDE.md`):
1. `reset_saturation_accumulator()`
2. `load_rotation_sequence(...)` → `calculate_linear_stokes`
3. `calculate_s3(...)` (popola `_WAV_INTENSITY_CACHE` per il rebasing Poincaré)
4. `generate_background_mask(S0)`
5. `align_reference_frame` → `align_poincare_ellipticity` (riassegnare S1, S3)
6. `calculate_dolp_aolp` + `calculate_retardance_and_fast_axis(target_folder=...)`

## Configurazione globale

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

import polarimetro as pol
from polarimetro import config as polcfg
from polarimetro import plotting as polplot
from polarimetro import umap_runner as polumap

polplot.apply_thesis_style()

DATASET = 'strati_v2'
CHANNEL = 'all'  # 'R' | 'G' | 'B' | 'all'
DOWNSAMPLE_FACTOR = 4
SAVE_PLOTS = True
OUTPUT_DIR = '../Images/generated'

RUN_SPECTRA = False
RUN_DEBUGGER = False
RUN_FULL_BATCH = False

TARGET_FOLDER = f'./raw/{DATASET}'
POL_SUBFOLDER = os.path.join(TARGET_FOLDER, 'pol')
WAV_SUBFOLDER = os.path.join(TARGET_FOLDER, 'wav')
WAVELENGTHS_CSV = './outputs/rgb_wavelengths.csv'
DARK_FRAME_PATH = './raw/dark.dng'

CHANNELS_RGB = {'R': 0, 'G': 1, 'B': 2}
ACTIVE_CHANNELS = list(CHANNELS_RGB.keys()) if CHANNEL == 'all' else [CHANNEL]

print(f"dataset={DATASET}, channels={ACTIVE_CHANNELS}, DS={DOWNSAMPLE_FACTOR}, swap={polcfg.is_waveplate_swapped(TARGET_FOLDER)}")

## Dispatcher analisi per dataset

Per ciascun dataset elenca le analisi da eseguire. Le celle stub sotto controllano `DATASET in dispatcher[...]` per gated execution.

In [ ]:
ANALYSES_PER_DATASET = {
    'strati_v2':         ['AB', 'C-delta', 'F', 'H', 'E-ext'],
    'lambdaquarti_50deg': ['AB', 'C-delta'],
    'lambdamezzi_50deg':  ['AB', 'C-delta'],
    'zucchero':          ['AB', 'C-aolp'],
    'barraon_v2':        ['AB'],
    'barraoff_v2':       ['AB'],
    'righello_v2':       ['AB'],
}

active_analyses = ANALYSES_PER_DATASET.get(DATASET, [])
print(f"analisi attive per {DATASET}: {active_analyses}")

## Load + Stokes — pipeline completa con cache npz

Per ogni canale attivo: carica RAW, calcola Stokes lineari, S3, maschere, applica allineamento (asse S3 + asse S2 / Poincaré), deriva DoLP/AoLP/δ/θ. Cache `outputs/stokes_<DATASET>_DS<DS>.npz` invalidata se cambia `DOWNSAMPLE_FACTOR` o `DATASET`.

In [ ]:
stokes_data = {}

if 'AB' in active_analyses:
    os.makedirs('./outputs', exist_ok=True)
    cache_path = f'./outputs/stokes_{DATASET}_DS{DOWNSAMPLE_FACTOR}.npz'

    cache_valid = False
    if os.path.exists(cache_path):
        try:
            npz = np.load(cache_path, allow_pickle=False)
            if int(npz['downsample_factor']) == DOWNSAMPLE_FACTOR \
                    and str(npz['dataset']) == DATASET \
                    and all(f'{ch}_S0' in npz.files for ch in ACTIVE_CHANNELS):
                for ch in ACTIVE_CHANNELS:
                    stokes_data[ch] = {
                        'S0': npz[f'{ch}_S0'],
                        'S1': npz[f'{ch}_S1'],
                        'S2': npz[f'{ch}_S2'],
                        'S3': npz[f'{ch}_S3'],
                        'bg_mask': npz[f'{ch}_bg_mask'].astype(bool),
                        'poincare_bg_mask': npz[f'{ch}_poincare_bg_mask'].astype(bool),
                        'sat_mask': (npz[f'{ch}_sat_mask'].astype(bool)
                                     if f'{ch}_sat_mask' in npz.files else None),
                        'DoLP': npz[f'{ch}_DoLP'],
                        'AoLP': npz[f'{ch}_AoLP'],
                        'delta': npz[f'{ch}_delta'],
                        'theta': npz[f'{ch}_theta'],
                        'wav_intensity': (npz[f'{ch}_wav_intensity']
                                          if f'{ch}_wav_intensity' in npz.files else None),
                        'channel_idx': CHANNELS_RGB[ch],
                    }
                cache_valid = True
                print(f"Cache loaded: {cache_path}")
            npz.close()
        except Exception as e:
            print(f"Cache read failed ({e}); recomputing.")

    if not cache_valid:
        for ch in ACTIVE_CHANNELS:
            ch_idx = CHANNELS_RGB[ch]
            print(f"\n=== Channel {ch} (idx={ch_idx}) ===")

            pol.reset_saturation_accumulator()

            angles, stack = pol.load_rotation_sequence(
                POL_SUBFOLDER, ch_idx,
                downsample_factor=DOWNSAMPLE_FACTOR,
                invert_angles=True,
                dark_frame_path=DARK_FRAME_PATH,
            )
            if angles is None or stack is None:
                raise RuntimeError(f"load_rotation_sequence failed for {DATASET}/{ch}")

            S0, S1, S2 = pol.calculate_linear_stokes(angles, stack)

            wavelength = polcfg.get_channel_wavelength(WAVELENGTHS_CSV, ch_idx)
            S3 = pol.calculate_s3(
                WAV_SUBFOLDER, ch_idx,
                downsample_factor=DOWNSAMPLE_FACTOR,
                wavelength=wavelength,
                dark_frame_path=DARK_FRAME_PATH,
            )

            bg_mask = pol.generate_background_mask(S0, downsample_factor=DOWNSAMPLE_FACTOR)
            S1, S2 = pol.align_reference_frame(S1, S2, bg_mask)
            S1, S3 = pol.align_poincare_ellipticity(
                S0, S1, S3, bg_mask,
                downsample_factor=DOWNSAMPLE_FACTOR,
            )

            DoLP, AoLP = pol.calculate_dolp_aolp(S0, S1, S2)
            delta_deg, theta_deg = pol.calculate_retardance_and_fast_axis(
                S0, S1, S2, S3, bg_mask,
                target_folder=TARGET_FOLDER,
            )

            sat_mask = pol.get_saturation_mask(DOWNSAMPLE_FACTOR)
            if sat_mask is not None:
                n_sat = int(sat_mask.sum())
                if n_sat:
                    frac = 100.0 * n_sat / sat_mask.size
                    print(f"Saturation: {n_sat} clipped blocks ({frac:.2f}%)")
                    for arr in (S1, S2, S3, DoLP, AoLP, delta_deg, theta_deg):
                        arr[sat_mask] = np.nan

            poincare_bg_mask = pol.get_poincare_bg_mask()
            if poincare_bg_mask is None:
                poincare_bg_mask = np.zeros_like(bg_mask)

            from polarimetro import stokes as polstokes
            wav_intensity = polstokes.get_wav_intensity_cache()

            stokes_data[ch] = {
                'S0': S0, 'S1': S1, 'S2': S2, 'S3': S3,
                'bg_mask': bg_mask,
                'poincare_bg_mask': poincare_bg_mask.copy(),
                'sat_mask': sat_mask,
                'DoLP': DoLP, 'AoLP': AoLP,
                'delta': delta_deg, 'theta': theta_deg,
                'wav_intensity': (wav_intensity.copy()
                                  if wav_intensity is not None else None),
                'channel_idx': ch_idx,
            }

            valid = bg_mask & np.isfinite(delta_deg)
            if valid.any():
                p5, p50, p95 = np.percentile(delta_deg[valid], [5, 50, 95])
                print(f"  {ch}: S0 shape={S0.shape}  "
                      f"delta_bg pctl [5,50,95]=[{p5:.1f}, {p50:.1f}, {p95:.1f}] deg")

        save_dict = {
            'downsample_factor': np.int32(DOWNSAMPLE_FACTOR),
            'dataset': np.array(DATASET),
        }
        for ch, d in stokes_data.items():
            for k, v in d.items():
                if v is None or k == 'channel_idx':
                    continue
                save_dict[f'{ch}_{k}'] = v
        np.savez_compressed(cache_path, **save_dict)
        print(f"\nCache saved: {cache_path}")

## AB — 9 mappe (display + autosave PDF/HTML)

Per ogni canale attivo, genera in stile pubblicazione le 9 mappe `S0, S1, S2, S3, DoLP, AoLP, δ, θ, mask`. `plt.show()` sempre; `plt.savefig()` solo se `SAVE_PLOTS=True` in `OUTPUT_DIR/<DATASET>/<CH>_<param>.pdf`. HTML plotly interattivi (parametri non-mask) in `OUTPUT_DIR/<DATASET>/interactive/<CH>_<param>.html`.

In [ ]:
if 'AB' in active_analyses and stokes_data:
    from matplotlib.colors import LinearSegmentedColormap
    from matplotlib.patches import Patch
    try:
        import plotly.graph_objects as go
        _PLOTLY_OK = True
    except ImportError:
        _PLOTLY_OK = False

    AB_PARAM_CONFIG = {
        'S0':    {'titolo': 'Intensità totale $S_0$',                 'unita': 'conteggi (u.a.)', 'cmap': None,       'vmin': None,    'vmax': None},
        'S1':    {'titolo': 'Parametro di Stokes $S_1$',              'unita': 'conteggi (u.a.)', 'cmap': 'bwr',      'vmin': 'sym99', 'vmax': 'sym99'},
        'S2':    {'titolo': 'Parametro di Stokes $S_2$',              'unita': 'conteggi (u.a.)', 'cmap': 'bwr',      'vmin': 'sym99', 'vmax': 'sym99'},
        'S3':    {'titolo': 'Parametro di Stokes $S_3$',              'unita': 'conteggi (u.a.)', 'cmap': 'bwr',      'vmin': 'sym99', 'vmax': 'sym99'},
        'DoLP':  {'titolo': 'Grado di polarizzazione lineare (DoLP)', 'unita': None,              'cmap': 'viridis',  'vmin': 0,       'vmax': 1},
        'AoLP':  {'titolo': "Angolo di polarizzazione lineare (AoLP)", 'unita': '°',              'cmap': 'twilight', 'vmin': -90,     'vmax': 90},
        'delta': {'titolo': r'Ritardo di fase $\delta$',              'unita': '°',              'cmap': 'twilight', 'vmin': 0,       'vmax': 360},
        'theta': {'titolo': r'Asse veloce $\theta$',                  'unita': '°',              'cmap': 'twilight', 'vmin': -90,     'vmax': 90},
        'mask':  {'titolo': 'Maschera di sfondo',                     'unita': None,              'cmap': 'gray',     'vmin': 0,       'vmax': 1},
    }
    _S0_CMAPS = {
        0: LinearSegmentedColormap.from_list('nero_rosso', ['black', 'red']),
        1: LinearSegmentedColormap.from_list('nero_verde', ['black', 'green']),
        2: LinearSegmentedColormap.from_list('nero_blu',   ['black', 'blue']),
    }

    def _resolve_limits(data, vmin_spec, vmax_spec):
        if vmin_spec == 'sym99':
            bound = float(np.nanpercentile(np.abs(data), 99))
            return -bound, bound
        return vmin_spec, vmax_spec

    def _mpl_cmap_to_plotly(cmap, n=64):
        cmap_obj = plt.get_cmap(cmap) if isinstance(cmap, str) else cmap
        scale = []
        for s in np.linspace(0.0, 1.0, n):
            r, g, b, _ = cmap_obj(float(s))
            scale.append([float(s), f"rgb({int(255*r)},{int(255*g)},{int(255*b)})"])
        return scale

    def _save_interactive_html(data, param, ch_idx, out_path):
        if not _PLOTLY_OK:
            return False
        cfg = AB_PARAM_CONFIG[param]
        cmap = _S0_CMAPS[ch_idx] if cfg['cmap'] is None else cfg['cmap']
        vmin, vmax = _resolve_limits(data, cfg['vmin'], cfg['vmax'])
        H, W = data.shape
        fig = go.Figure(data=go.Heatmap(
            z=data,
            colorscale=_mpl_cmap_to_plotly(cmap),
            zmin=vmin, zmax=vmax,
            hovertemplate='x: %{x}<br>y: %{y}<br>valore: %{z:.4g}<extra></extra>',
            colorbar=dict(title=cfg['unita'] or ''),
        ))
        fig.update_layout(
            title=cfg['titolo'],
            xaxis=dict(scaleanchor='y', constrain='domain'),
            yaxis=dict(autorange='reversed'),
            width=min(1000, 80 + W),
            height=min(900, 80 + H),
            margin=dict(l=40, r=40, t=60, b=40),
        )
        try:
            fig.write_html(out_path, include_plotlyjs='cdn', full_html=True)
            return True
        except Exception as e:
            print(f"  (avviso: HTML plotly non scritto per {out_path}: {e})")
            return False

    def _make_figure(param, data, ch_idx):
        cfg = AB_PARAM_CONFIG[param]
        if param == 'mask' and data.ndim == 3:
            H, W, _ = data.shape
            aspect = H / W
            fig_w = 3.35
            fig, ax = plt.subplots(figsize=(fig_w + 0.2, fig_w * aspect))
            ax.imshow(data, aspect='equal')
            ax.set_title(cfg['titolo'], pad=6)
            ax.axis('off')
            handles = [
                Patch(facecolor='#bfbfbf', edgecolor='none', label='entrambe (S0)'),
                Patch(facecolor='#0040ff', edgecolor='none', label='XOR (wav debug)'),
                Patch(facecolor='#ff0000', edgecolor='none', label='nessuna (sample)'),
            ]
            ax.legend(handles=handles, loc='lower right', fontsize=6,
                      framealpha=0.85, handlelength=1.2,
                      borderpad=0.3, labelspacing=0.25)
            return fig
        cmap = _S0_CMAPS[ch_idx] if cfg['cmap'] is None else cfg['cmap']
        vmin, vmax = _resolve_limits(data, cfg['vmin'], cfg['vmax'])
        H, W = data.shape
        aspect = H / W
        fig_w = 3.35
        fig, ax = plt.subplots(figsize=(fig_w + 0.7, fig_w * aspect))
        im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax, aspect='equal')
        ax.set_title(cfg['titolo'], pad=6)
        ax.axis('off')
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
        if cfg['unita']:
            cbar.set_label(cfg['unita'])
        return fig

    out_dir = os.path.join(OUTPUT_DIR, DATASET)
    interactive_dir = os.path.join(out_dir, 'interactive')
    if SAVE_PLOTS:
        os.makedirs(out_dir, exist_ok=True)
        os.makedirs(interactive_dir, exist_ok=True)

    AB_PARAMS = ['S0', 'S1', 'S2', 'S3', 'DoLP', 'AoLP', 'delta', 'theta', 'mask']

    for ch in ACTIVE_CHANNELS:
        d = stokes_data[ch]
        ch_idx = d['channel_idx']
        wav_mean = d['wav_intensity'] / 2.0 if d['wav_intensity'] is not None else None
        mask_rgb = polplot.mask_overlay_rgb(
            d['S0'], d['bg_mask'], d['poincare_bg_mask'], wav_mean=wav_mean)
        data_map = {
            'S0': d['S0'], 'S1': d['S1'], 'S2': d['S2'], 'S3': d['S3'],
            'DoLP': d['DoLP'], 'AoLP': d['AoLP'],
            'delta': d['delta'], 'theta': d['theta'],
            'mask': mask_rgb,
        }
        for param in AB_PARAMS:
            data = data_map[param]
            if data is None:
                continue
            fig = _make_figure(param, data, ch_idx)
            pdf_path = os.path.join(out_dir, f"{ch}_{param}.pdf") if SAVE_PLOTS else None
            polplot.save_and_show(fig, pdf_path, show=True, fmt='pdf')
            plt.close(fig)
            if SAVE_PLOTS and param != 'mask':
                html_path = os.path.join(interactive_dir, f"{ch}_{param}.html")
                _save_interactive_html(data, param, ch_idx, html_path)

## C-aolp — UMAP scatter + AoLP map + AoLP histogram

Embedding UMAP sparso a risoluzione nativa, colorato per AoLP (cmap viridis, autoscala 1-99 pct). Default per zucchero.

In [ ]:
if 'C-aolp' in active_analyses:
    import time as _time

    UMAP_SPARSE_STRIDE = 20
    HIST_BINS_UMAP = 180
    CACHE_DIR_UMAP = './outputs'
    os.makedirs(CACHE_DIR_UMAP, exist_ok=True)

    def _sparse_grid_mask(shape, stride):
        m = np.zeros(shape, dtype=bool)
        m[::stride, ::stride] = True
        return m

    def compute_or_load_umap_cache(ch, sd, stride=UMAP_SPARSE_STRIDE):
        cache_path = os.path.join(CACHE_DIR_UMAP,
                                  f"umap_{DATASET}_{ch}_cache.npz")
        if os.path.exists(cache_path):
            with np.load(cache_path) as d:
                ok = ('delta_deg' in d.files and 'axis_conf_min' in d.files
                      and float(d['axis_conf_min']) == polumap.UMAP_AXIS_CONFIDENCE_MIN)
                if ok:
                    print(f"[cache] carico {cache_path}")
                    return (d['embedding'].copy(),
                            d['aolp_deg'].copy(),
                            d['delta_deg'].copy(),
                            d['valid_indices'].copy(),
                            tuple(int(x) for x in d['S0_shape']))
            print(f"[cache] {cache_path} obsoleto, ricomputo")
        S0 = sd['S0']; S1 = sd['S1']; S2 = sd['S2']; S3 = sd['S3']
        DoLP = sd['DoLP']; AoLP = sd['AoLP']
        delta = sd['delta']; theta = sd['theta']
        bg_mask = sd['bg_mask']; sat_mask = sd.get('sat_mask')
        base_valid = polumap.build_validity_mask(
            S0, DoLP, bg_mask, sat_mask=sat_mask, theta_deg=theta)
        valid_mask = base_valid & _sparse_grid_mask(S0.shape, stride)
        features, valid_indices = polumap.build_feature_matrix(
            S0, S1, S2, S3, DoLP, valid_mask, feature_mode='no_delta')
        print(f"  pixel validi: {features.shape[0]}")
        if features.shape[0] < 100:
            print("  too few valid pixels, skip")
            return None
        t0 = _time.time()
        embedding = polumap.fit_umap(features)
        print(f"  UMAP fit: {_time.time()-t0:.1f}s")
        np.savez_compressed(
            cache_path, embedding=embedding, aolp_deg=AoLP,
            delta_deg=delta, valid_indices=valid_indices,
            S0_shape=np.array(S0.shape, dtype=np.int64),
            axis_conf_min=np.float32(polumap.UMAP_AXIS_CONFIDENCE_MIN))
        print(f"[cache] salvato {cache_path}")
        return embedding, AoLP, delta, valid_indices, S0.shape

    def export_umap_panels(spec, embedding, valid_indices, S0_shape,
                           dataset_label, channel_label,
                           hist_bins=HIST_BINS_UMAP):
        out = os.path.join(OUTPUT_DIR, dataset_label,
                           spec['export_subdir'], channel_label)
        os.makedirs(out, exist_ok=True)
        cmap = spec['cmap']
        vmin, vmax = spec['vmin'], spec['vmax']
        norm = plt.Normalize(vmin=vmin, vmax=vmax)
        H, W = S0_shape
        value_map = spec['value_map']
        value_valid = spec['value_valid']
        extend = spec['cbar_extend']

        aspect_img = H / W
        fig_w = 3.35
        fig_h = fig_w * aspect_img
        fig, ax = plt.subplots(figsize=(fig_w + 0.7, fig_h))
        im = ax.imshow(value_map, cmap=cmap, vmin=vmin, vmax=vmax, aspect='equal')
        ax.set_title(spec['map_title_short'], pad=6)
        ax.axis('off')
        cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03, extend=extend)
        cb.set_label(spec['unit_label'])
        polplot.save_and_show(fig, os.path.join(out, spec['export_map_file']),
                              show=SAVE_PLOTS)
        plt.close(fig)

        fig, ax = plt.subplots(figsize=(3.35 + 0.7, 3.35))
        ax.scatter(embedding[:, 0], embedding[:, 1],
                   c=value_valid, cmap=cmap, vmin=vmin, vmax=vmax,
                   s=2.0, alpha=0.85)
        ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
        ax.set_title('UMAP embedding', pad=6)
        cb = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap),
                          ax=ax, fraction=0.046, pad=0.03, extend=extend)
        cb.set_label(spec['unit_label'])
        polplot.save_and_show(fig, os.path.join(out, 'umap_scatter.pdf'),
                              show=SAVE_PLOTS)
        plt.close(fig)

        hmin, hmax = spec['hist_range']
        fig, ax = plt.subplots(figsize=(3.35, 2.2))
        finite = value_valid[np.isfinite(value_valid)]
        counts, edges = np.histogram(finite, bins=hist_bins, range=(hmin, hmax))
        centers = 0.5 * (edges[:-1] + edges[1:])
        widths = np.diff(edges)
        bar_colors = [cmap(norm(c)) for c in centers]
        ax.bar(centers, counts, width=widths, color=bar_colors,
               edgecolor='none', align='center')
        edge = spec['hist_edge_exclude']
        if edge > 0.0:
            inner = (centers > hmin + edge) & (centers < hmax - edge)
            ymax = (float(counts[inner].max())
                    if (inner.any() and counts[inner].size)
                    else (float(counts.max()) if counts.size else 1.0))
            ax.axvspan(hmin, hmin + edge, color='gray', alpha=0.08)
            ax.axvspan(hmax - edge, hmax, color='gray', alpha=0.08)
            ax.set_ylim(0, ymax * 1.25)
        else:
            ymax = float(np.percentile(counts, 99)) if counts.size else 1.0
            ax.set_ylim(0, ymax * 1.15)
        ax.set_xlim(hmin, hmax)
        ax.set_xlabel(spec['unit_label'])
        ax.set_ylabel('# pixel validi')
        ax.set_title(spec['hist_title'].replace('(c) ', ''), pad=6)
        ax.grid(True, axis='y', linestyle=':', alpha=0.4)
        polplot.save_and_show(fig, os.path.join(out, spec['export_hist_file']),
                              show=SAVE_PLOTS)
        plt.close(fig)
        return out

    UMAP_CACHE_BY_CHANNEL = {}
    for _ch in ACTIVE_CHANNELS:
        print(f"\n=== UMAP {DATASET} / {_ch} (color_by=aolp) ===")
        _sd = stokes_data.get(_ch)
        if _sd is None:
            print(f"  [{_ch}] skip: mancano stokes_data (eseguire Load+Stokes prima)")
            continue
        _result = compute_or_load_umap_cache(_ch, _sd)
        if _result is None:
            continue
        _embedding, _aolp, _delta, _valid_indices, _S0_shape = _result
        UMAP_CACHE_BY_CHANNEL[_ch] = _result
        _spec = polumap.color_spec('aolp', aolp_deg=_aolp,
                                   delta_deg=_delta,
                                   valid_indices=_valid_indices)
        _out = export_umap_panels(_spec, _embedding, _valid_indices, _S0_shape,
                                  DATASET, _ch)
        print(f"  [{_ch}/aolp] esportato: {_out}/")

## C-delta — UMAP scatter + δ map + δ histogram

Stesso fit di C-aolp (cache condivisa), colorato per retardance δ (cmap ciclica twilight 0-360, bande di esclusione ±20°). Default per strati/λ-quarti/λ-mezzi.

In [ ]:
# TODO: analysis_cells
if 'C-delta' in active_analyses:
    pass

## E-ext — Istogrammi δ pubblicabili (strati)

PDF + HTML interattivo con barre colorate twilight. Solo `strati_v2`.

In [ ]:
# TODO: analysis_cells
if 'E-ext' in active_analyses:
    pass

## F — Slice diagonale δ multistrato (publication-style)

Tre pannelli (mappa δ con banda evidenziata | profilo 1D etichettato 1L-5-1R | fit through-origin con unwrap per-side). Solo `strati_v2`.

In [ ]:
# TODO: analysis_cells
if 'F' in active_analyses:
    pass

## H — Fit retardance vs strati + dispersione 1/λ²

Punti retardance per strato (3 canali RGB) + fit lineare attraverso l'origine + correzione dispersiva 1/λ². Solo `strati_v2`.

In [ ]:
# TODO: analysis_cells
if 'H' in active_analyses:
    pass

## Strumenti opzionali

Celle gated dai flag `RUN_*` in cima al notebook (off-by-default).

### G0 — Stima centroidi spettrali RGB

Esegue `final_monochrome_approx` (legacy) e aggiorna `outputs/rgb_wavelengths.csv` + PDF spettrale.

In [ ]:
# TODO: optional_cells
if RUN_SPECTRA:
    pass

### I — Debugger pixel-per-pixel

Ispettore interattivo del fit Stokes pixel-per-pixel (animazione intensità vs angolo).

In [ ]:
# TODO: optional_cells
if RUN_DEBUGGER:
    pass

### FULL BATCH — 7 dataset × 3 canali × 9 parametri

Esegue AB su tutti i dataset attivi nel dispatcher; 189 PDF + 189 HTML in `OUTPUT_DIR/<dataset>/`.

In [ ]:
# TODO: optional_cells
if RUN_FULL_BATCH:
    pass